# Reading PHYSLITE files using the `PHYSLITESchema`

`NanoEvents` is a `coffea` utility to wrap NTuple structures (such as the ATLAS PHYSLITE format) into a single awkward array with appropriate object methods (such as Lorentz vector methods), cross references, and nested objects. All are by default accessed from the input ROOT TTree via `uproot`.  

The interpretation of the TTree data is configurable via schema objects, which are community-supplied for various input file types. Relevant schemas for this training:
1. `BaseSchema`, which provides a simple representation of the input TTree, where each branch is available verbatim as `events.branch_name`. Any branches that `uproot` supports at "full speed" (i.e. that are fully split and either flat or single-jagged) can be read by this schema.
2. `PHYSLITESchema`, for the ATLAS PHYSLITE derivations.

The schema code, along with their specialized methods, can be found here: https://github.com/CoffeaTeam/coffea/tree/master/src/coffea/nanoevents

In this demo we will read the content of a PHYSLITE file using the `PHYSLITESchema`. The usage and benefits of the schema will be demonstrated.

In [ ]:
import coffea
print("coffea version: ", coffea.__version__)
import awkward as ak
print("awkward version: ", ak.__version__)
from coffea.nanoevents import NanoEventsFactory, PHYSLITESchema
import numpy as np
import matplotlib.pyplot as plt
import warnings

Let's load a PHYSLITE file containing simulated event data describing the production the Higgs boson and its decay via a pair of Z bosons to a four lepton final state. Those data are part of the recent ATLAS open data release and can be found at DOI:[10.7483/OPENDATA.ATLAS.Z2J9.709J](http://doi.org/10.7483/OPENDATA.ATLAS.Z2J9.709J).

More information about our data release at [this article](https://atlas.cern/Updates/News/Open-Data-Research) and the ATLAS open data portal: https://opendata.atlas.cern/

You can download those files at your local machine or you can stream them directly. For this demo we will use the [XCache](https://slateci.io/XCache/) service of the UChicago Analysis Facility.

In [ ]:
# HZZ -> 4l sample

# local
# file_path = "/Users/iason/DAOD_PHYSLITE.38191712._000001.pool.root.1"

# stream
# file_path = "root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000001.pool.root.1"

# XCache
file_path = "root://xcache.af.uchicago.edu:1094//root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000001.pool.root.1"

To make the loading lighter, we will use the `filter_name` function, in order to load only the variables needed. Then we will use `coffea`'s `NanoEventsFactory.from_root` to load the file. The variables are grouped in data structures called "containers" and the convention to reference them is `<ContainerName>.<VariableName>`. Consider looking at the [`from_root`](https://coffeateam.github.io/coffea/api/coffea.nanoevents.NanoEventsFactory.html#coffea.nanoevents.NanoEventsFactory.from_root) class method to see all optional arguments.

All the containers and variables stored in the open data PHYSLITE files are listed in: https://atlas-physlite-content-opendata.web.cern.ch/. For your reference, the equivalent internal webpage for general PHYSLITE files is: https://atlas-physlite-content.web.cern.ch/. 

_Some warnings will appear as we're still working with `uproot` and the `PHYSLITESchema` to correctly interpret all the variables stored in PHYSLITE._

In [ ]:
warnings.filterwarnings(
    "ignore",
    message="Skipping ",
    category=UserWarning,
)

events = NanoEventsFactory.from_root(
    {file_path: "CollectionTree"}, # all the event variables are stored in the TTree called CollectionTree
    schemaclass=PHYSLITESchema, # tell NanoEventsFactory.from_root that you read a PHYSLITE file
    mode="virtual", # use "eager" if you want to load all the events on the spot
).events()

The `events` object is an awkward array, which at its top level is a record array* with one record for each "collection", where a collection is a grouping of objects of the seme type, based on the naming conventions of `PHYSLITESchema`. We try to correspond the collections in awkward arrays with the containers found in PHYSLITE.

\* _Record array_ a is a specialized type of array that allows you to store elements with mixed data types that can be accessed using attribute notation, e.g. `array.x` [[1](https://numpy.org/doc/stable/reference/generated/numpy.recarray.html)], [[2](https://awkward-array.org/doc/main/reference/generated/ak.Record.html)].

For example, in the file we opened there are the collections:

In [ ]:
events.fields

For each collection, we can list the variables that have been loaded (TBranches). For example, let's look at the variables of the `Electrons`:

In [ ]:
events.Electrons.fields

The data in a PHYSLITE file are highly structured but jagged. In other words, in each event there are different number of objects in each collection, thus the awkward arrays are not rectilinear but _awkward_. We can see that if we actually request to compute the objects of the `Electrons` collection.

In [ ]:
ak.materialize(events.Electrons)

## Slicing

One of the most common operations you will found yourself doing is event selection or array _slicing_. We can practice this by first defining an awkward (boolean) array by a logical operation. Then we can use this array as an argument to slice our events.

For example, let's request events with at least two electrons:

In [ ]:
# this cell might fail on the first time -- if so, run twice or restart the kernel

# define boolean array
selection_2e = ak.num(events.Electrons, axis=1) > 1

# print the selected events
events[selection_2e]

Notice how the number of events in the awkward array has been reduced.

In [ ]:
# print the second of the selected events
events[selection_2e][1]

In [ ]:
# print the leading electron pt of the selected events
events.Electrons[selection_2e][:, 0].pt # in MeV
# events[selection_2e][:, "Electrons"][:, "pt"][:, 0] # another uglier way to write this

Next, we will demonstrate something more complicated. This is an example of the power of the schemas along with vector operations*. We will use a utility method [`delta_r`](https://coffeateam.github.io/coffea/api/coffea.nanoevents.methods.vector.LorentzVector.html#coffea.nanoevents.methods.vector.LorentzVector.delta_r) to calculate the distance $\Delta R = \sqrt{\Delta\eta^2+\Delta\phi^2}$ between two LorentzVector objects. In particular, we will calculate the distance between the leading and sub-leading electrons in each event. Of course, we need to make use we use events that have at least two electrons, thus we will utilize the `selection_2e` from above.

To plot a histogram we will just use `matplotlib` for the moment. You might also want to try the [`hist`](https://github.com/scikit-hep/hist) library, which you will learn about at the [Introduction to hist](https://indico.cern.ch/event/1376945/timetable/?view=standard#33-introduction-to-hist) lecture.

_\* `coffea` still uses the internal [`coffea.nanoevents.methods.vector`](https://coffeateam.github.io/coffea/modules/coffea.nanoevents.methods.vector.html) but it will soon switch to the [`vector`](https://github.com/scikit-hep/vector) library, which will hopefully be feature-compatible._

In [ ]:
# distance between leading and sub-leading electron in every event
dr = events.Electrons[selection_2e][:, 0].delta_r(events.Electrons[selection_2e][:, 1])

# plot
plt.hist(dr, bins=50, range=(0, 5))
plt.xlabel(r"$\Delta R(e_0, e_1)$")
plt.ylabel("Events")
plt.show()

Let's now use the schema to calculate the invariant mass of a group of particles. In particular, calculate and plot the invariant mass of the four leading electrons of each event.

<details>
<summary><b>Hint</b></summary>

Use the [mass](https://coffeateam.github.io/coffea/api/coffea.nanoevents.methods.vector.LorentzVector.html#coffea.nanoevents.methods.vector.LorentzVector.mass) attribute.
</details>

In [ ]:
# your code here

<details>
<summary><b>Answer</b></summary>

```python
# define boolean array to select events with 4 electrons
selection_4e = ak.num(events.Electrons, axis=1) == 4

# calculate the invariant mass of the 4 electrons
mass = (events.Electrons[selection_4e][:, 0] + 
        events.Electrons[selection_4e][:, 1] + 
        events.Electrons[selection_4e][:, 2] + 
        events.Electrons[selection_4e][:, 3]).mass

# plot
plt.hist(mass/1000, bins=10, range=(100, 150))
plt.xlabel(r"$m(4e)$ [GeV]")
plt.ylabel("Events")
plt.show()
```
</details>

## Element Links

Some collections contain information which is contextually relevant to another collection. For example, in PHYSLITE files we store the `BTagging_AntiKt4EMPFlow` collection which contains information regarding the probabilities of the flavour origin of the objects of the `Jets` collection.

The information is referenced and accessed by the `ElementLinks`. `ElementLinks` are like pointers that help point to specific objects in a collection. They have a structure similar to a Python dictionary with a key (`m_persKey`) to identify the collection they point to and a value (`m_persIndex`) to identify the index of the element in that collection. 

On the above example, each object of the `Jets` collection (i.e. each jet) holds a variable called `btaggingLink`, which points to the objects of the `BTagging_AntiKt4EMPFlow` collection. The index is used to identify which exactly `BTagging_AntiKt4EMPFlow` object corresponds to each jet.

In [ ]:
events.Jets.btaggingLink.fields

### Naive Linking

We're currently working with `coffea` and the `PHYSLITESchema` to enable automatic cross-reference (_ElementLinking_) between collections. However, in simple cases we can achieve the linking, without actually using the `ElementLinks`.

When there is **one-to-one** correspondence between the objects of two collections, we can link those without the need to resolve the actual links. For our example, each object of the `Jets` collection is linked to one object of the `BTagging_AntiKt4EMPFlow` collection.

We will write a function to perform that linking. We will use the [DL1dv01](https://ftag.docs.cern.ch/algorithms/taggers/dl1/) flavour-tagging algorithm and information about its calibration for b-tagging can be found [here](https://ftag.docs.cern.ch/recommendations/algs/r22-preliminary/#working-point-definition-for-dl1dv01).

In [ ]:
def calculate_jets_DL1dv01(events):
    
    BTagging = events.BTagging_AntiKt4EMPFlow
    
    f_c = 0.018
    DL1dv01 = BTagging.DL1dv01_pb/(f_c*BTagging.DL1dv01_pc + (1-f_c)*BTagging.DL1dv01_pu)
    DL1dv01 = np.log(DL1dv01)

    return DL1dv01

We will use the above function to assign a new variable to the `Jets` collection. One can assign new variables to the arrays, with some caveats:

* Assignment must use `events["path", "to", "name"] = value`.
* Assignment to a sliced events won't be accessible from the original variable.

In [ ]:
# assign new variable to the collection
events['Jets', 'DL1dv01'] = calculate_jets_DL1dv01(events)

events.Jets.fields

Let's finally use the new variable to make a selection. We will plot the leading jet transverse momentum ($p_T$) for events that have at least one b-tagged jet at the 77% efficiency working point.

In [ ]:
# boolean array -- events with at least one b-jet at 77% WP
selection_1bjet = ak.sum(events.Jets.DL1dv01 > 2.456, axis=-1) > 0

# plot the leading jet pt
plt.hist(events.Jets[selection_1bjet][:, 0].pt/1000, bins=50, range=(0, 500))
plt.xlabel(r"$p_T(j_0)$ [GeV]")
plt.ylabel("Events")
plt.show()

## Preview: upcoming coffea features

You just read a file into `NanoEvents` with the virtual-array backend. The features previewed below concern how coffea will *describe*, *preprocess*, and *execute* over such datasets in upcoming releases.

The cells below preview features that are **not yet released** — they live in open *draft* pull requests against the coffea repository. Each demo is **guarded**: it detects whether the feature is present (by import / signature introspection, never by version number, since these are unreleased branches) and prints a gentle note instead of raising if it is missing. This section is therefore safe to "Run All" in the default environment.

To actually exercise the demos, launch one of the preview environments defined in `pixi.toml`:

```bash
pixi run -e preview jupyter lab           # pydantic dataset-tools extensions: PRs #1579, #1600, #1601
pixi run -e preview-compute jupyter lab    # the coffea.compute execution refactor: PR #1470
```

Those environments install coffea straight from the PR branches, so the exact API may drift before release.

In [ ]:
# --- Feature detection for the preview demos below ---------------------------
# These upcoming features live in *unreleased* draft PRs, so we never rely on a
# version number; each is detected by import / signature introspection. When a
# feature is absent the demo cells print a gentle note instead of raising, so
# this whole section is safe to "Run All" in the default environment.
import importlib.util
import inspect
from pathlib import Path

import coffea
from coffea.dataset_tools import preprocess as _preprocess


def _has_param(func, name):
    try:
        return name in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False


HAS_PP_BACKENDS = _has_param(_preprocess, "backend")             # draft PR #1579
HAS_PP_METADATA = _has_param(_preprocess, "metadata_extractor")  # draft PR #1600
HAS_MUTABLE_STEPS = importlib.util.find_spec("coffea.dataset_tools.mutable_steps") is not None  # draft PR #1601
HAS_COMPUTE = importlib.util.find_spec("coffea.compute") is not None  # draft PR #1470


def preview_note(feature, pr):
    print(
        f"[preview] '{feature}' is not available in this coffea build ({coffea.__version__}).\n"
        f"          It ships in draft PR {pr}. To try it, launch a preview environment:\n"
        f"            pixi run -e preview jupyter lab           # pydantic dataset-tools extensions (#1579/#1600/#1601)\n"
        f"            pixi run -e preview-compute jupyter lab    # coffea.compute execution refactor (#1470)"
    )


# A small, network-free sample so the preview demos run wherever this repo is
# checked out (they fall back gracefully if it is missing).
_preview_file = Path("../columnar/data/SMHiggsToZZTo4L.root")
_preview_fileset = {
    "demo": {"files": {str(_preview_file): "Events"}, "metadata": {"xsec": 1.0}}
}

print(f"coffea {coffea.__version__}")
for _flag in ["HAS_PP_BACKENDS", "HAS_PP_METADATA", "HAS_MUTABLE_STEPS", "HAS_COMPUTE"]:
    print(f"  {_flag} = {globals()[_flag]}")

### 1. Pydantic dataset specifications

*Released in coffea 2026.7 (PR #1528) — the foundation the previews build on.*

Filesets can now be expressed as validated `pydantic` models (`DataGroupSpec` / `DatasetSpec` / `ROOTFileSpec`, ...). Malformed filesets fail fast with clear errors, and the models carry form and metadata around for the tools below.

In [ ]:
# 1. Pydantic dataset specifications (released in coffea 2026.7, PR #1528)
# The classic "dict-in / dict-out" fileset still works, but datasets can now be
# expressed as *validated* pydantic models, catching malformed filesets early.
from coffea.dataset_tools import ModelFactory, DatasetSpec

spec = ModelFactory.dict_to_datasetspec(_preview_fileset["demo"])
print("type:", type(spec).__name__, "| is DatasetSpec:", isinstance(spec, DatasetSpec))
print("validated metadata:", dict(spec.metadata))
print("file specs:", [type(fs).__name__ for fs in spec.files.values()])
# round-trip back to a plain dict when a legacy API needs one
_roundtrip = ModelFactory.datasetspec_to_dict(spec)

### 2. Non-dask preprocessing backends — draft PR #1579

Released `preprocess()` builds a **dask-awkward** graph to discover file chunks. PR #1579 adds a `backend=` switch (`"iterative"`, `"futures"`, `"dask"`) so preprocessing can run with **no dask dependency**, plus a dedicated `preprocess_rntuple()` for RNTuple inputs. The backend classes (`IterativeBackend`, `FuturesBackend`, ...) are explicitly designed to plug into the `coffea.compute` refactor below.

In [ ]:
# 2. Non-dask preprocessing backends  (draft PR #1579)
from coffea.dataset_tools import preprocess

if HAS_PP_BACKENDS and _preview_file.exists():
    available, report = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",  # or "futures"; "dask" reproduces the legacy path
        skip_bad_files=True,
    )
    finfo = list(available["demo"]["files"].values())[0]
    print("preprocessed with the dask-free 'iterative' backend")
    print("  steps discovered:", finfo["steps"])
    print("  num_entries:", finfo["num_entries"])
elif not HAS_PP_BACKENDS:
    preview_note("preprocess(backend=...)", "#1579")
else:
    print("[preview] sample file not found; skipping the live run.")

### 3. User-supplied metadata extraction — draft PR #1600

Computing per-dataset quantities such as the sum of generator weights normally means an extra pass over the files. PR #1600 adds `metadata_extractor` (called once per file on the open handle) and `metadata_reducer` (called once per dataset) hooks to `preprocess()`, folding that work into the preprocessing pass.

In [ ]:
# 3. User-supplied metadata extraction during preprocessing  (draft PR #1600)
from coffea.dataset_tools import preprocess

if HAS_PP_METADATA and _preview_file.exists():
    def per_file(file_handle):
        # runs once per file, on the open uproot file handle
        return {"nentries": int(file_handle["Events"].num_entries)}

    def per_dataset(per_file_meta):
        # reduce the per-file dicts into dataset-level metadata
        return {"nentries_total": sum(m["nentries"] for m in per_file_meta.values())}

    available, _ = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",
        metadata_extractor=per_file,
        metadata_reducer=per_dataset,
        skip_bad_files=True,
    )
    print("dataset metadata after extraction:", dict(available["demo"]["metadata"]))
elif not HAS_PP_METADATA:
    preview_note("preprocess(metadata_extractor=..., metadata_reducer=...)", "#1600")
else:
    print("[preview] sample file not found; skipping the live run.")

### 4. Adaptive / resizable steps — draft PR #1601

Fixed step sizes over- or under-shoot when chunk cost varies. This **prototype** adds a resizable step generator whose size can be renegotiated mid-stream through the generator `.send()` channel (the same channel `coffea.compute`'s `Computable.gen_steps` uses), plus a `run_adaptive_steps` driver governed by a `WallTimeStepPolicy`. The API is explicitly marked unstable.

In [ ]:
# 4. Adaptive / resizable steps  (draft PR #1601, prototype -- API may change)
if HAS_MUTABLE_STEPS:
    from coffea.dataset_tools.mutable_steps import resizable_steps

    gen = resizable_steps(0, 1_000, 200)
    produced = [next(gen)]
    try:
        while True:
            # after the first chunk, ask the generator to shrink the step to 100
            produced.append(gen.send(100))
    except StopIteration:
        pass
    print("resizable_steps, shrunk mid-stream via .send(100):")
    print(" ", produced)

    # Higher-level driver, operating on a preprocessed pydantic DatasetSpec:
    print(
        "\nHigher-level API (illustrative):\n"
        "    from coffea.dataset_tools.mutable_steps import (\n"
        "        iter_dataset_steps, run_adaptive_steps, WallTimeStepPolicy)\n"
        "    policy = WallTimeStepPolicy(target_seconds=30)\n"
        "    total = run_adaptive_steps(dataset_spec, work_fn, step_size=100_000, policy=policy)"
    )
else:
    preview_note("coffea.dataset_tools.mutable_steps", "#1601")

### 5. A unified execution protocol: `coffea.compute` — draft PR #1470

The largest change on the horizon. PR #1470 introduces `coffea.compute`, replacing the `Processor` / `Executor` / `Runner` trio with a single `Backend` **protocol**. Work is expressed as a `Computable` (a `Dataset` mapped through a function via `.map_steps`), handed to any backend's `.compute()`, which returns a non-blocking `Task` exposing `.result()`, `.partial_result()`, `.wait()`, and `.cancel()`. The preprocessing backends (#1579) and resizable steps (#1601) are stepping stones toward this unified interface. The one-liner it enables:

```python
with ThreadedBackend() as backend:
    total = backend.compute(dataset.map_steps(process)).result()
```

This backend is a genuine work in progress, so the demo below is guarded to show the protocol *shape* even where it does not yet fully execute.

In [ ]:
# 5. A unified execution protocol: coffea.compute  (draft PR #1470, WIP)
# PR #1470 replaces the Processor/Executor/Runner trio with a single `Backend`
# protocol. A `Computable` (a Dataset mapped through a function via .map_steps)
# is handed to any backend's .compute(), returning a non-blocking Task with
# .result()/.partial_result(). This is what tutorial scaleout could look like
# once the refactor lands:
if HAS_COMPUTE:
    from coffea.compute.data import Dataset, File, ContextDataset
    from coffea.compute.backends.threaded import ThreadedBackend

    dataset = Dataset(
        files=[File(path=str(_preview_file), steps=[(0, 50_000), (50_000, 100_000)])],
        metadata=ContextDataset(dataset_name="demo", cross_section=None),
    )

    def count(events):  # a plain callable *is* the processor
        return len(events)

    computable = dataset.map_steps(count)
    print(f"built a Computable with {len(computable)} work element(s)")
    print(
        "the target one-liner:\n"
        "    with ThreadedBackend() as backend:\n"
        "        total = backend.compute(computable).result()"
    )
    try:
        with ThreadedBackend() as backend:
            total = backend.compute(computable).result()
        print("result:", total)
    except Exception as exc:  # coffea.compute is a work-in-progress preview
        print(
            f"[preview] coffea.compute did not execute here yet ({type(exc).__name__}); "
            "the protocol shape above is the point of this WIP preview."
        )
else:
    preview_note("coffea.compute", "#1470")